# Cek Dataset Seed Keterampilan Berbicara

Notebook ini sengaja dibuat compact: cukup memuat ringkasan rerata/SD, uji beda antarkelompok, dan perubahan pre-post.

Ekspektasi data contoh:
- pretes kelas eksperimen dan kontrol kurang lebih sama;
- postes kelas eksperimen naik ringan;
- kenaikannya tidak dibuat ekstrem.


In [ ]:
import csv, math
from statistics import mean, stdev
from pathlib import Path

base = Path('data/field_test')
files = {
    'pre': base / 'keterampilan_berbicara_pretes_seed.csv',
    'post': base / 'keterampilan_berbicara_postes_seed.csv',
}

def load(kind):
    score_col = f'{kind}_nilai_akhir'
    with files[kind].open(newline='', encoding='utf-8') as f:
        return [
            {'id': r['id'], 'kelompok': r['kelompok'], 'score': float(r[score_col]), 'is_seed': int(r['is_seed'])}
            for r in csv.DictReader(f)
        ]

def by_group(rows):
    return {g: [r['score'] for r in rows if r['kelompok'] == g] for g in ('eksperimen', 'kontrol')}

def desc(values):
    return {'n': len(values), 'mean': mean(values), 'sd': stdev(values)}

def cohen_d(a, b):
    sa, sb = stdev(a), stdev(b)
    pooled = math.sqrt(((len(a)-1)*sa**2 + (len(b)-1)*sb**2) / (len(a)+len(b)-2))
    return (mean(a) - mean(b)) / pooled

def normal_p_from_t(t):
    return math.erfc(abs(t) / math.sqrt(2))

def welch(a, b):
    va, vb = stdev(a)**2, stdev(b)**2
    se = math.sqrt(va/len(a) + vb/len(b))
    t = (mean(a) - mean(b)) / se
    return {'diff': mean(a) - mean(b), 't': t, 'p_approx': normal_p_from_t(t), 'd': cohen_d(a, b)}

def paired_change(pre, post, group):
    pre_map = {r['id']: r['score'] for r in pre if r['kelompok'] == group}
    post_map = {r['id']: r['score'] for r in post if r['kelompok'] == group}
    changes = [post_map[i] - pre_map[i] for i in pre_map]
    return {'n': len(changes), 'mean_change': mean(changes), 'sd_change': stdev(changes)}

def seed_counts(rows):
    return {'n_asli': sum(r['is_seed'] == 0 for r in rows), 'n_seed': sum(r['is_seed'] == 1 for r in rows)}

pre, post = load('pre'), load('post')
pre_g, post_g = by_group(pre), by_group(post)
summary = {
    'flag_pre_per_mahasiswa': seed_counts(pre),
    'flag_post_per_mahasiswa': seed_counts(post),
    'pre_eksperimen': desc(pre_g['eksperimen']),
    'pre_kontrol': desc(pre_g['kontrol']),
    'post_eksperimen': desc(post_g['eksperimen']),
    'post_kontrol': desc(post_g['kontrol']),
    'uji_pre_eksperimen_vs_kontrol': welch(pre_g['eksperimen'], pre_g['kontrol']),
    'uji_post_eksperimen_vs_kontrol': welch(post_g['eksperimen'], post_g['kontrol']),
    'perubahan_eksperimen': paired_change(pre, post, 'eksperimen'),
    'perubahan_kontrol': paired_change(pre, post, 'kontrol'),
}
for label, stats in summary.items():
    print(label)
    for k, v in stats.items():
        print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')


: 